# M21 · Deep-learning pickers: probability curves, pretrained weights and PocketQuake's picker

**Week 3 · about 7.8 h** · before this: M10, M20

> **한국어 요약** 딥러닝 피커(PhaseNet 계열)는 표본마다 P파와 S파가 도달했을 확률을 곡선으로 내놓습니다. 이 곡선에서 피킹을 골라내는 함수를 직접 만들고, 두 가지 사전 학습 가중치의 피킹을 M20에서 여러분이 읽은 피킹과 비교합니다. 이어서 PocketQuake의 피킹 단계를 세 번 실행하고, 그 피킹 파일에서 초동 극성 표를 만듭니다.

**You will**

- Turn a probability curve into picks
- Compare two pretrained PhaseNet models with your picks
- Run PocketQuake's picker three times and compare the runs
- Read first motions from pick files

**New words:** deep-learning picker (딥러닝 위상 검출기), pick probability (피킹 확률), pick threshold (피킹 임계값), model weights (모형 가중치), pretrained model (사전 학습 모형), blinding (edge samples; 가장자리 표본 제외), systematic error (계통 오차), deterministic computation (결정론적 계산) · [all terms](../../../docs/glossary.md)

**Optional reading:** Mousavi & Beroza (2022), an overview; Zhu & Beroza (2019), sections 1-3

Log any AI help in `work/ai_log/M21.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M21.yaml](../../../.sgtl/criteria/M21.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download of M03
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks of M20 (work/picks/)
WEIGHTS = {"stead": "2", "original": "2"}  # two pretrained PhaseNet weights (name: version)
THRESHOLD = 0.3                           # smallest probability of a pick
MIN_GAP_S = 1.0                           # picks of one phase closer than this (s): only the higher one counts
WINDOW_S = (-2.0, 40.0)                   # picks of this earthquake lie here (s after the origin)
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster (2-40 characters: a-z, 0-9, _)
PQ_MEM_GB = 8.0                           # memory cap of each PocketQuake run (GB); the picker needs about 3.5
PQ_MIN_PROB_CHANGED = 0.5                 # Part 4: the one PocketQuake setting you change (default 0.3)
FM_MIN_WEIGHT = 0.3                       # Part 5: smallest polarity size that counts as a first motion
FM_MIN_PROB = 0.5                         # Part 5: smallest P-pick probability whose polarity counts
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M21")

import hashlib
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display


def show_figure(figure, filename):
    """Save and display the same static figure, including in an archived notebook."""
    path = Path("figures") / filename
    path.parent.mkdir(exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    display(Image(filename=str(path)))
    plt.close(figure)
from obspy import Stream, UTCDateTime, read

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
# SeisBench looks for its weights in this folder; 'bin/sgtl-install seisbench' put the course's weights there.
os.environ.setdefault("SEISBENCH_CACHE_ROOT", str(st.pq.sgtl_home() / "seisbench"))
print("SeisBench weights folder:", os.environ["SEISBENCH_CACHE_ROOT"])

## Part 0 · Your data and your picks (about 15 min)

You need three things from earlier work:

- your STP download of `sangju2019` (M03);
- your M20 picks, `work/picks/M20_manual_picks.csv`;
- the course tools: `bin/sgtl-install tools` and `bin/sgtl-install seisbench` (read each dry run, then add
  `--apply`).

The cell reads the earthquake (KMA catalog, UTC), your picks and your records.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EVENT_FOLDER = ORIGIN.strftime("%Y%m%d%H%M%S")         # STP's name for this earthquake (UTC origin time)
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {float(event['latitude']):.2f} N, "
      f"{float(event['longitude']):.2f} E, depth {float(event['depth_km']):g} km, ML {float(event['ml']):.1f}")

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. This module compares the pickers with your own picks: finish "
                            "task M20-T5 of module M20 first (its picker saves the file).")
manual = pd.read_csv(PICKS_CSV)
manual["time"] = [UTCDateTime(t) for t in manual["time_utc"]]
print(f"\nyour manual picks ({len(manual)} rows, {manual['station'].nunique()} stations):")
print(manual[["station", "phase", "channel", "time_utc", "uncertainty_s", "polarity"]].to_string(index=False))

records = Stream()
for name, rows in manual.groupby("station"):
    band = str(rows["channel"].iloc[0])[:2]
    net, sta = name.split(".")
    records += read(str(SAC_DIR / band / f"*.{net}.{sta}.{band}?.sac"))
records.sort(keys=["network", "station", "channel"])
print(f"\n{len(records)} traces of {len(manual['station'].unique())} stations read, e.g.")
print(records[:3])

## Part 1 · What a deep-learning picker gives you (about 45 min)

A **deep-learning picker** is a neural network trained on records whose arrivals analysts had picked. Training
set its millions of numbers, the **model weights**. A network with weights that someone already trained is a
**pretrained model**; other weights make a different picker.

The course pins two PhaseNet weights (`WEIGHTS`):

- `stead`: trained on STEAD, a million records from many regions; PocketQuake's default;
- `original`: trained by Zhu and Beroza on records from Northern California.

The cell loads both and compares each file's **SHA-256 fingerprint** with the course's pinned value. A
fingerprint is a 64-character code computed from a file's bytes; one changed byte changes it.

In [ ]:
import torch

torch.set_num_threads(1)                         # inference on one core (shared-server rule)
try:
    import seisbench
    import seisbench.models as sbm
except ImportError as exc:
    raise ImportError(f"SeisBench cannot load its models in this environment ({exc}). Run 'sgtl doctor' in a "
                      "terminal and tell your mentor.") from None
print(f"SeisBench {seisbench.__version__}, PyTorch {torch.__version__}, threads used: {torch.get_num_threads()}")

pinned = {}
for line in st.data.repo_path("install", "checksums.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 3 and parts[0].startswith("phasenet/"):
        pinned[parts[0]] = parts[1]
models = {}
for name, version in WEIGHTS.items():
    for kind in ("json", "pt"):
        rel = f"phasenet/{name}.{kind}.v{version}"
        path = Path(seisbench.cache_model_root) / rel
        if not path.is_file():
            raise FileNotFoundError(f"{path} is missing. In a terminal: bin/sgtl-install seisbench --apply")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        verdict = "matches the course's pin" if pinned.get(rel) == digest else "DOES NOT MATCH the pin: tell your mentor"
        print(f"{rel}: SHA-256 {digest[:16]}... {verdict}")
    models[name] = sbm.PhaseNet.from_pretrained(name, version_str=version)
    models[name].eval()
for name, model in models.items():
    print(f"\n{name}: {model.weights_docstring.strip().splitlines()[0]}")
    print(f"  component order {model.component_order}, normalization {model.norm!r}, "
          f"settings that come with the weights: {model.default_args}")

**PhaseNet** reads 30 s of a three-component record at 100 samples per second. For every sample it gives three
numbers that add up to 1: how likely a P wave, an S wave or only noise arrives there. Each series is a
**pick-probability curve**. SeisBench's `annotate` slides PhaseNet along a long record in overlapping pieces.
Run it on the station you picked first.

In [ ]:
first_station = manual.sort_values("time").iloc[0]["station"]
one = records.select(network=first_station.split(".")[0], station=first_station.split(".")[1])
curves = models["stead"].annotate(one)
print(curves)
print(f"\nrecord:   {one[0].stats.starttime} to {one[0].stats.endtime} ({one[0].stats.npts} samples)")
p_curve = curves.select(channel="*_P")[0]
s_curve = curves.select(channel="*_S")[0]
print(f"P curve:  {p_curve.stats.starttime} to {p_curve.stats.endtime} ({p_curve.stats.npts} samples)")
total = np.sum([tr.data for tr in curves], axis=0)
print(f"P + S + noise lies between {total.min():.4f} and {total.max():.4f} at every sample")

z = one.select(component="Z")[0]
fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)
axes[0].plot(z.times(reftime=ORIGIN), z.data - np.mean(z.data), color="k", lw=0.5)
axes[0].set_ylabel(f"{z.stats.channel} (counts)")
axes[1].plot(p_curve.times(reftime=ORIGIN), p_curve.data, color="tab:blue", lw=0.9, label="P")
axes[1].plot(s_curve.times(reftime=ORIGIN), s_curve.data, color="tab:red", lw=0.9, label="S")
for _, row in manual[manual["station"] == first_station].iterrows():
    for ax in axes:
        ax.axvline(row["time"] - ORIGIN, color="0.5", ls="--", lw=0.8)
axes[1].set_ylabel("Probability")
axes[1].set_xlabel("Time after the origin (s)")
axes[1].set_xlim(-5.0, 30.0)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8, title=f"{first_station}, stead weights "
           "(dashed: your picks)", title_fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "probability_curves.png", dpi=150)

The curves start 2.5 s after the record and end 2.5 s before it. Near the ends of each piece the network sees
too little record, so SeisBench drops its output there (**blinding**). Each piece is scaled before it enters the
network, so raw counts work as well as velocity.

**Watch out:** PhaseNet learned mostly from seismometer records. On accelerometer (HG) records its picks may be
weaker or slightly off; check them by eye.

A picker keeps the **local maxima** of a curve (samples higher than both neighbors) that reach a **pick
threshold**. It also keeps no two picks of one phase closer than a minimum gap; otherwise small humps on one
arrival would each become a pick. A threshold too low lets noise through as a **false pick**; one too high, or a
gap too long, loses real arrivals.

### M21-T1 · From a probability curve to picks · core · about 60 min

Write `picks_from_probability(prob, delta, threshold, min_gap_s)`. It returns `(times_s, values)`: the pick times
in seconds after the first sample and their probabilities, sorted by time. Apply the gap from the highest pick
down: a pick stays when no higher pick that stays lies closer than `min_gap_s`.

Example: `prob = [0, 0.2, 0.6, 0.3, 0.7, 0.1, 0, 0.4, 0]`, `delta = 0.5`, `threshold = 0.35`, `min_gap_s = 1.5`
give `(array([2. , 3.5]), array([0.7, 0.4]))`.

Not allowed: scipy.signal, scipy.ndimage, SeisBench, obspy.signal and torch.

In [ ]:
# Before you code: in the example above, which picks do you get with min_gap_s = 0.5 instead?
st.predict("M21-T1", """

""")

In [ ]:
def picks_from_probability(prob, delta, threshold, min_gap_s):
    """Picks in one pick-probability curve: peaks at or above a threshold, never two closer than min_gap_s.

    prob      : 1-D numpy array, one probability per sample (it is not changed)
    delta     : sample spacing (s)
    threshold : smallest probability of a pick
    min_gap_s : two picks closer than this (s) cannot both stay; the higher one stays
    returns   : (times_s, values), two 1-D float arrays of the same length, sorted by time: each pick's sample
                number times delta (s after the first sample), and prob at those samples.
    A peak is a sample higher than the sample before it and the sample after it; the first and the last sample are
    never peaks. A pick is a peak whose probability is at least threshold.
    """
    # Step 1: the peaks: samples higher than the sample before and the sample after (never the two end samples).
    # Step 2: keep the peaks at or above the threshold.
    # Step 3: visit them from the highest down, and keep one only when no kept pick lies closer than min_gap_s.
    # Step 4: sort what you kept by time; give back the times in seconds and the probabilities.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T1: write your code here")

In [ ]:
st.check("M21-T1", func=picks_from_probability)

The whole 500-s P curve of Part 1: where it rises above 0.1, then your function with a few thresholds and gaps.

In [ ]:
above = p_curve.data >= 0.1
edges = np.flatnonzero(np.diff(above.astype(int)) == 1) + 1
print(f"the P curve rises above 0.1 at {edges.size} places:")
for k in edges:
    stop = k + int(np.argmax(~above[k:])) if (~above[k:]).any() else p_curve.stats.npts
    print(f"  {p_curve.times(reftime=ORIGIN)[k]:7.2f} s after the origin, highest value {p_curve.data[k:stop].max():.3f}")
print()
for thr, gap in ((0.05, 0.0), (0.1, 0.0), (0.3, 0.0), (0.3, MIN_GAP_S), (0.6, MIN_GAP_S)):
    secs, probs = picks_from_probability(p_curve.data, p_curve.stats.delta, thr, gap)
    after = [round(float(p_curve.stats.starttime + x - ORIGIN), 2) for x in secs]
    print(f"threshold {thr:4.2f}, gap {gap:3.1f} s: {len(secs):3d} picks at (s after the origin) {after[:8]}"
          + (" ..." if len(after) > 8 else ""))



**A peak is not yet an arrival.** Top: the whole P curve; bottom: the closest pair of candidate peaks. Circles
are peaks above the threshold with no gap; stars are the picks your function keeps with the gap.

In [ ]:
visual_t = p_curve.times(reftime=ORIGIN)
no_gap_s, no_gap_p = picks_from_probability(p_curve.data, p_curve.stats.delta, THRESHOLD, 0.0)
kept_s, kept_p = picks_from_probability(p_curve.data, p_curve.stats.delta, THRESHOLD, MIN_GAP_S)
no_gap_t = np.asarray(no_gap_s) + float(p_curve.stats.starttime - ORIGIN)
kept_t = np.asarray(kept_s) + float(p_curve.stats.starttime - ORIGIN)
if len(no_gap_t) > 1:
    closest = int(np.argmin(np.diff(no_gap_t)))
    zoom_center = float(np.mean(no_gap_t[closest:closest + 2]))
else:
    zoom_center = float(visual_t[int(np.argmax(p_curve.data))])
fig, axes = plt.subplots(2, 1, figsize=(9, 5.1), constrained_layout=True)
for ax in axes:
    ax.plot(visual_t, p_curve.data, color="0.3", lw=0.8, label="P probability")
    ax.axhline(THRESHOLD, color="tab:red", ls="--", lw=0.8, label="Probability threshold")
    ax.scatter(no_gap_t, no_gap_p, facecolors="none", edgecolors="tab:orange", s=60, label="No minimum gap")
    ax.scatter(kept_t, kept_p, marker="*", color="tab:blue", s=70, label=f"Gap {MIN_GAP_S:g} s")
    ax.set(xlabel="Time after origin (s)", ylabel="Probability", ylim=(-0.03, 1.05))
axes[0].set_title("Whole record: candidate times")
axes[0].legend(fontsize=8, ncol=2)
axes[1].set(xlim=(zoom_center - 0.6, zoom_center + 0.6), title="Zoom: neighboring peaks on one probability curve")
show_figure(fig, "M21_probability_peak_selection.png")

**Explain:** Without a gap your function finds four picks: what are they, and why does the 1-s gap leave two?

## Part 2 · Picks of two pretrained models (about 75 min)

`annotate` returns an ObsPy Stream of curves, one trace per station and phase: SeisBench calls it the
**annotations**. Channel names end with `_P`, `_S` or `_N` (noise), and each curve has its own start time.

### M21-T2 · The best pick of every station · core · about 60 min

Write `best_picks(annotations, threshold, min_gap_s, start, end)`. It returns a DataFrame with the most probable
P and S pick of every station between `start` and `end` (the docstring lists the columns).

Example: a station has P picks 7 s (0.9), 22 s (0.95) and 393 s (0.97) after the origin. With a window from -2
to 40 s, its P row is the pick at 22 s, with 0.95.

Not allowed: scipy.signal, scipy.ndimage, SeisBench and obspy.signal.

In [ ]:
# Before you code: will the two models' P picks lie within one sample (0.01 s) of each other at your stations?
st.predict("M21-T2", """

""")

In [ ]:
def best_picks(annotations, threshold, min_gap_s, start, end):
    """The most probable P and S pick of every station inside [start, end].

    annotations     : obspy Stream from a SeisBench model's annotate; channel names end with _P, _S or _N
                      (noise curves give no picks)
    threshold       : smallest probability of a pick
    min_gap_s       : the minimum gap of picks_from_probability (s)
    start, end      : obspy UTCDateTime limits of the window (both included)
    returns         : pandas DataFrame with columns station (network.station, e.g. KS.ZZA1), phase ("P" or
                      "S"), time (obspy UTCDateTime) and probability: for every station and phase with a pick
                      inside the window, the one pick with the highest probability; sorted by station, then phase.
    The time of sample k of a curve is that curve's own start time plus k * delta.
    """
    # Step 1: go through the P and S curves (their channel names end with _P or _S); skip the noise curves.
    # Step 2: run your picks_from_probability on each curve; turn its times into UTC with the curve's start time.
    # Step 3: keep the picks between start and end, and of these the most probable one.
    # Step 4: one row per station and phase; sort by station, then phase.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T2: write your code here")

In [ ]:
st.check("M21-T2", func=best_picks, dataset=DATASET)

Both models on all your stations, with the window of the parameters cell. The table goes to
`results/M21_model_picks.csv`.

In [ ]:
window = (ORIGIN + WINDOW_S[0], ORIGIN + WINDOW_S[1])
per_model = []
for name, model in models.items():
    rows = best_picks(model.annotate(records), THRESHOLD, MIN_GAP_S, *window)
    rows.insert(0, "model", name)
    per_model.append(rows)
model_picks = pd.concat(per_model, ignore_index=True)
shown = model_picks.assign(after_origin_s=[round(t - ORIGIN, 3) for t in model_picks["time"]])
print(shown.drop(columns="time").to_string(index=False))
model_picks.assign(time=model_picks["time"].astype(str)).to_csv(Path(RESULT_DIR) / "M21_model_picks.csv",
                                                                 index=False)

**Explain:** How far apart are the two models' P picks at your stations, and is one model always later?

## Part 3 · Your picks against the models' picks (about 45 min)

To compare two sets of picks, match them by station and phase (and by earthquake, when there are several). A
pick without a partner matters too: a model may miss an arrival you saw, or find one you did not pick.

A **systematic error** has the same sign and size at every station, like a model that always picks 0.03 s late.
More stations do not average it away. The median of the differences shows it. The **median absolute
deviation** (MAD) shows the scatter around that median, and one wild pick cannot dominate it.

### M21-T3 · Matching two sets of picks · core · about 45 min

Write `compare_pick_sets(a, b)`. It returns one row per pick in `a` or `b`, with `time_a`, `time_b`, `diff_s`
(`time_b` minus `time_a`, in s) and `status` (`"both"`, `"only_a"` or `"only_b"`). A table that holds one pick
twice must raise a `ValueError`.

Example: `a` has XX.ST01 P and S; `b` has XX.ST01 P, 0.015 s later, and XX.ST02 P. The result has three rows:
XX.ST01 P (both, +0.015 s), XX.ST01 S (only_a), XX.ST02 P (only_b).

In [ ]:
# Before you code: you compare your picks (a) with a model (b). Which sign of diff_s means the model picked
# later than you?
st.predict("M21-T3", """

""")

In [ ]:
def compare_pick_sets(a, b):
    """Match the picks of two pick tables by their key and give the time differences b - a.

    a, b    : pandas DataFrames with columns station, phase, time (obspy UTCDateTime), maybe event_id and maybe
              other columns; not changed
    key     : (event_id, station, phase) when both tables have an event_id column, otherwise (station, phase)
    returns : DataFrame with one row per key found in a or in b: the key columns, time_a and time_b (the times,
              None where that table has no such pick), diff_s (time_b - time_a in s; NaN unless both have it)
              and status ("both", "only_a" or "only_b"); sorted by the key columns
    raises  : ValueError when a key appears twice in a or in b
    """
    # Step 1: the key: event_id (only when both tables have that column), station and phase.
    # Step 2: stop with a ValueError when a key appears twice in either table.
    # Step 3: an outer join of the two tables on the key, keeping each table's time under its own name.
    # Step 4: status, empty places as None, diff_s in seconds for keys in both; sort by the key.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T3: write your code here")

In [ ]:
st.check("M21-T3", func=compare_pick_sets)

Your picks (a) against each model (b): every pair, then the median and the MAD per model and phase. The figure
shows the differences against your P time (a stand-in for distance).

In [ ]:
mine = manual[["station", "phase", "time"]]
summary, matched = [], {}
for name in models:
    table = compare_pick_sets(mine, model_picks[model_picks["model"] == name])
    matched[name] = table
    print(f"\n{name}:")
    print(table.assign(time_a=table["time_a"].astype(str)).drop(columns="time_b").round(3).to_string(index=False))
    for phase, part in table[table["status"] == "both"].groupby("phase"):
        d = part["diff_s"].to_numpy()
        summary.append({"model": name, "phase": phase, "matched": d.size, "median_s": np.median(d),
                        "mad_s": np.median(np.abs(d - np.median(d))), "min_s": d.min(), "max_s": d.max()})
summary = pd.DataFrame(summary)
print("\n" + summary.round(3).to_string(index=False))
summary.to_csv(Path(RESULT_DIR) / "M21_pick_differences.csv", index=False)

p_time = {r["station"]: r["time"] - ORIGIN for _, r in manual[manual["phase"] == "P"].iterrows()}
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
for ax, phase in zip(axes, ("P", "S")):
    for name, color in zip(models, ("tab:blue", "tab:orange")):
        part = matched[name][(matched[name]["phase"] == phase) & (matched[name]["status"] == "both")]
        ax.plot([p_time[s] for s in part["station"]], part["diff_s"], "o", color=color, ms=4, label=name)
    ax.axhline(0.0, color="0.5", lw=0.8)
    ax.set_title(f"{phase} picks", fontsize=9)
    ax.set_xlabel("Your P pick (s after the origin)")
axes[0].set_ylabel("Model minus your pick (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "model_minus_manual.png", dpi=150)

**Explain:** Is each model's P offset from your picks systematic? Compare its median with its MAD.

**Unmatched picks count too.** The bars count matched pairs, picks only you made and picks only the model made.
An unmatched model pick is a case to inspect, not always a false arrival.

In [ ]:
visual_rows = []
for name, compared in matched.items():
    for phase in ("P", "S"):
        part = compared[compared["phase"] == phase]
        visual_rows.append({"label": f"{name} {phase}",
                            **{status: int((part["status"] == status).sum()) for status in ("both", "only_a", "only_b")}})
visual_counts = pd.DataFrame(visual_rows)
fig, ax = plt.subplots(figsize=(8, 3.7), constrained_layout=True)
visual_bottom = np.zeros(len(visual_counts))
for field, label, color in (("both", "Matched", "tab:blue"), ("only_a", "Manual only", "tab:orange"),
                            ("only_b", "Model only", "0.6")):
    ax.bar(visual_counts["label"], visual_counts[field], bottom=visual_bottom, color=color, label=label)
    visual_bottom += visual_counts[field].to_numpy()
ax.set(xlabel="Model and phase", ylabel="Number of picks", title="Your manual–model comparison, including unmatched cases")
ax.legend(fontsize=8, ncol=3)
show_figure(fig, "M21_matched_and_unmatched.png")

## Part 4 · PocketQuake's picker, run three times (about 90 min)

**PocketQuake** runs picking as one stage of its chain: stations, waveforms, picking, location, relocation,
mechanisms. Through `st.pq` its picker is **PhaseNet+** (EQNet), a PhaseNet-family network that also gives every
P pick a first-motion polarity. PocketQuake works on a **cluster**: your dataset registered under a short name,
`PQ_SLUG`.

Test a production tool like your own code, with two more runs:

1. **An unchanged rerun.** Is the result **deterministic**, the same byte for byte? If not, any later comparison
   mixes your change with that scatter.
2. **One setting changed.** PocketQuake reads its pick threshold from the **environment variable**
   `PNPLUS_MIN_PROB` (default 0.3). A program started from Python inherits Python's environment, so the cell sets
   it, runs, and removes it.

Each `st.pq` call first prints a dry run; `apply=True` runs it on one core. A run overwrites the picks folder, so
the cells copy it after each run.

### M21-T4 · PocketQuake's pick files, three runs · core · about 90 min

Write `read_pocketquake_picks(folder)`: all picks of a PocketQuake picks folder in one table (the docstring lists
the columns). Then make the three runs with the cells below and fill in `pq_runs` as the comments of its cell
say.

The check recomputes every fingerprint from your copied files and looks for your runs in the run log,
`work/.sgtl/runs.jsonl`.

In [ ]:
# Before you run: will the unchanged rerun write the same pick files, byte for byte?
st.predict("M21-T4", """

""")

In [ ]:
def read_pocketquake_picks(folder):
    """All picks of a PocketQuake picks folder (<event id>_picks.csv files) in one table.

    folder  : str or pathlib.Path of the folder. Each earthquake has one file <event id>_picks.csv with the
              columns Event_ID, Network, Station, Phase, Time (ISO text in UTC), Probability and, from a picker
              that measures them, Polarity and Amplitude. Other files in the folder are not pick files.
    returns : pandas DataFrame with every pick of every *_picks.csv file and the columns event_id (text, such as
              "20000101000000"), station (network.station), phase, time (obspy UTCDateTime), probability (float)
              and, only when the files have a Polarity column, polarity (float, NaN for S picks). Never make up a
              column the files do not have; other columns may stay.
    """
    # Step 1: the pick files only: names ending in _picks.csv, in name order.
    # Step 2: read them with pandas, with Event_ID kept as text (it is a name, not a number), and stack them.
    # Step 3: build the columns of the docstring; add polarity only when the files have a Polarity column.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T4: write your code here")

Run 1. Register your dataset as a PocketQuake cluster (once: read the dry run, then apply). Then run the stages
stations, waveforms and picking, about 2 minutes on one core.

In [ ]:
cluster_file = st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py"
if cluster_file.is_file():
    print(f"Cluster {PQ_SLUG!r} is registered already ({cluster_file.name}); nothing to stage.")
else:
    st.pq.stage(DATASET, PQ_SLUG)                             # dry run: read what it would do

In [ ]:
if not cluster_file.is_file():
    assert st.pq.stage(DATASET, PQ_SLUG, apply=True) == 0
st.pq.run(PQ_SLUG, "stations", "picking", cores=1, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "stations", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
PQ_PICKS = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG / "picks"
COPIES = Path(RESULT_DIR) / "pq_picks"


def keep_copy(label):
    """Copy PocketQuake's picks folder to results/pq_picks/<label> (the next run overwrites the original)."""
    target = COPIES / label
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(PQ_PICKS, target)
    print(f"copied {len(list(target.glob('*_picks.csv')))} pick files to {target}")
    return target


first_dir = keep_copy("first")

Run 2: the picking stage again, nothing changed.

In [ ]:
assert st.pq.run(PQ_SLUG, "picking", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_dir = keep_copy("rerun")

Run 3: one setting changed. This is the line of PocketQuake's settings (in your copy) that reads the threshold:

In [ ]:
config_lines = (st.pq.pipeline_dir() / "pipeline" / "config.py").read_text(encoding="utf-8").splitlines()
print("\n".join(f"config.py:{n}: {line}" for n, line in enumerate(config_lines, 1) if "PNPLUS_MIN_PROB" in line))

In [ ]:
os.environ["PNPLUS_MIN_PROB"] = str(PQ_MIN_PROB_CHANGED)      # PocketQuake reads it when the run starts
try:
    status = st.pq.run(PQ_SLUG, "picking", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True)
finally:
    del os.environ["PNPLUS_MIN_PROB"]                         # every later run gets the default again
assert status == 0
changed_dir = keep_copy(f"min_prob_{PQ_MIN_PROB_CHANGED:g}")

The run log: one line per run, with the fingerprints of the files it wrote.

In [ ]:
log_lines = st.data.repo_path("work", ".sgtl", "runs.jsonl").read_text(encoding="utf-8").splitlines()
for line in log_lines[-3:]:
    run = json.loads(line)
    print(run["ts"], f"{run['stage_from']} -> {run['through']}", f"{run['seconds']:.0f} s", run["products"])

In [ ]:
# Fill in pq_runs from your three runs:
#     pq_runs = {"reader": read_pocketquake_picks, "slug": PQ_SLUG, "first": <copy of the first run>,
#                "rerun": <copy of the rerun>, "changed": <copy of the changed run>,
#                "min_prob_changed": PQ_MIN_PROB_CHANGED,
#                "fingerprints": {"first": <prints of the first copy>, "rerun": <prints of the rerun copy>}}
# The prints of a copy: a dict from the name of every *_picks.csv file in it (such as "20000101000000_picks.csv")
# to the SHA-256 of that file's bytes (hashlib.sha256(...).hexdigest(), 64 hex letters). Compute them from the
# files after the copy cells: the check recomputes every one.
# YOUR CODE HERE
raise NotImplementedError("M21-T4: write your code here")
print({k: (v.__name__ if callable(v) else v) for k, v in pq_runs.items() if k != "fingerprints"})

In [ ]:
st.check("M21-T4", answer=pq_runs)

The two copies file by file (from your fingerprints), then pick by pick with your `compare_pick_sets` (the key
now includes the earthquake):

In [ ]:
prints = pq_runs["fingerprints"]
same_files = sorted(n for n in prints["first"] if prints["rerun"].get(n) == prints["first"][n])
print(f"pick files with the same SHA-256 in the first run and the rerun: {len(same_files)} of "
      f"{len(prints['first'])} (the rerun has {len(prints['rerun'])})")
first_run = read_pocketquake_picks(first_dir)
print(f"first run: {len(first_run)} picks of {first_run['event_id'].nunique()} earthquakes, "
      f"{first_run['station'].nunique()} stations; probabilities {first_run['probability'].min():.3f} to "
      f"{first_run['probability'].max():.3f}")
for label, folder in (("rerun", rerun_dir), ("changed", changed_dir)):
    table = compare_pick_sets(first_run, read_pocketquake_picks(folder))
    moved = int(((table["status"] == "both") & (table["diff_s"].abs() > 0.0)).sum())
    print(f"\nfirst run against the {label} run: {table['status'].value_counts().to_dict()}, moved picks: {moved}")
    print(table.groupby(["event_id", "status"]).size().unstack(fill_value=0).to_string())

**Explain:** At the higher threshold, which earthquakes lost the most picks, and why did the mainshock lose so
few?

## Part 5 · First motions from a pick file (about 60 min)

A **focal mechanism** (M51) can be found from the P first motions at many stations. Programs such as SKHASH
search for the fault orientations that fit the ups and downs. In PocketQuake's pick files the `Polarity` column
holds a number from −1 to 1. Its sign is the direction (positive is up); its size says how sure the picker is.
PocketQuake ignores polarities smaller than 0.3, and those of P picks with a probability below 0.5.

### M21-T5 · First motions from a pick table · core · about 60 min

Write `first_motions(picks, min_weight, min_prob)`: the first motion of every P pick in a table from your
`read_pocketquake_picks`. It is

- `U` when the polarity is at least `min_weight`;
- `D` when the polarity is at most −`min_weight`;
- `?` otherwise, and whenever the pick's probability is below `min_prob`.

Example: with `min_weight = 0.3` and `min_prob = 0.5`, P picks with (polarity, probability) = (0.8, 0.9),
(−0.3, 0.7), (0.2, 0.9), (0.9, 0.4) give U, D, ?, ?.

In [ ]:
# The colleague's pick file and helper. Run this cell (it writes the file and defines the helper); do not edit it.
# The file holds the colleague's picks of the mainshock, made with PhaseNet and the 'stead' weights, which is
# what PocketQuake's picking stage runs when it is started without a picker option (picker_weights='stead').
colleague_dir = Path(RESULT_DIR) / "colleague_picks"
colleague_dir.mkdir(exist_ok=True)
theirs = model_picks[model_picks["model"] == "stead"]
pd.DataFrame({"Event_ID": EVENT_FOLDER, "Network": theirs["station"].str.split(".").str[0],
              "Station": theirs["station"].str.split(".").str[1], "Phase": theirs["phase"],
              "Time": theirs["time"].astype(str), "Probability": theirs["probability"].round(3)}
             ).to_csv(colleague_dir / f"{EVENT_FOLDER}_picks.csv", index=False)


def colleague_first_motions(folder):
    """The colleague's helper: first motions (U = up, D = down) of the P picks of a PocketQuake picks folder."""
    picks = pd.concat([pd.read_csv(f, dtype={"Event_ID": str}) for f in sorted(Path(folder).glob("*_picks.csv"))])
    p = picks[picks["Phase"] == "P"]
    weight = p["Polarity"] if "Polarity" in p.columns else pd.Series(0.0, index=p.index)
    return pd.DataFrame({"station": p["Network"] + "." + p["Station"],
                         "first_motion": np.where(weight.fillna(0.0) >= 0.0, "U", "D")})


print(f"wrote {colleague_dir / f'{EVENT_FOLDER}_picks.csv'}")

In [ ]:
# Before you code: how many U, D and ? will colleague_first_motions give for the colleague's folder?
st.predict("M21-T5", """

""")

Now measure: the columns of the colleague's file, and what the helper makes of it. The last lines come from
PocketQuake's own code and are about pickers and polarities.

In [ ]:
print("columns of the colleague's file:", pd.read_csv(next(colleague_dir.glob("*_picks.csv"))).columns.tolist())
print("the colleague's helper:", colleague_first_motions(colleague_dir)["first_motion"].value_counts().to_dict())
pipeline_code = st.pq.pipeline_dir() / "pipeline"
for rel, words in (("config.py", ("SEISBENCH_MODELS =", "EQNET_MODELS =", "picker_weights: str")),
                   ("clusters/_base.py", ("picker_weights=",)),
                   ("core/focal_mechanism.py", ('"Polarity" not in', "phasenet_plus picking run"))):
    for number, line in enumerate((pipeline_code / rel).read_text(encoding="utf-8").splitlines(), 1):
        if any(w in line for w in words):
            print(f"{rel}:{number}: {line.strip()}")

In [ ]:
def first_motions(picks, min_weight, min_prob):
    """First motions (U, D or ?) of the P picks of a pick table made by read_pocketquake_picks.

    picks      : pandas DataFrame with station, phase, probability, maybe event_id, and polarity; not changed
    min_weight : smallest size of a polarity value that counts as a first motion
    min_prob   : smallest P-pick probability whose polarity counts
    returns    : DataFrame with one row per P pick (S picks give none): event_id (only when picks has it),
                 station and first_motion: "U" when polarity >= min_weight, "D" when polarity <= -min_weight,
                 "?" otherwise, for a missing polarity, and whenever probability < min_prob; sorted by event_id
                 (when present), then station
    raises     : ValueError when the table has no polarity column
    """
    # Step 1: no polarity column means no polarity at all: raise a ValueError (never guess).
    # Step 2: the P picks only.
    # Step 3: U when the polarity reaches min_weight, D when it reaches -min_weight, else a question mark (also for
    #         NaN, and for every pick whose probability is below min_prob).
    # Step 4: the key columns and first_motion, sorted.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T5: write your code here")

In [ ]:
st.check("M21-T5", func=first_motions)

The check also gave your function 3 tables without a `polarity` column. It had to stop with a `ValueError`:
such a table holds no first motions.

Your function on the colleague's folder and on your first PocketQuake run, then PhaseNet+'s first motions of the
mainshock next to your M20 polarities:

In [ ]:
try:
    first_motions(read_pocketquake_picks(colleague_dir), FM_MIN_WEIGHT, FM_MIN_PROB)
    print("your function gave first motions for the colleague's file")
except ValueError as exc:
    print("your function stopped on the colleague's file:", exc)

motions = first_motions(first_run, FM_MIN_WEIGHT, FM_MIN_PROB)
print("\nPhaseNet+ first motions, all earthquakes of your first run:")
print(motions.groupby(["event_id", "first_motion"]).size().unstack(fill_value=0).to_string())
side = (manual[manual["phase"] == "P"][["station", "polarity"]].rename(columns={"polarity": "yours"})
        .merge(motions[motions["event_id"] == EVENT_FOLDER][["station", "first_motion"]], on="station", how="left"))
side["agree"] = side["yours"] == side["first_motion"]
print("\nmainshock: your first motions (M20) and PhaseNet+'s")
print(side.to_string(index=False))
print(f"agree at {int(side['agree'].sum())} of {len(side)} stations")

**Explain:** What would a mechanism program have made of the colleague's table, and what does PocketQuake's own
mechanism stage do with such a file?

**Watch out:** a pick file without a `Polarity` column holds no first motions. Never fill a missing polarity with
0 or "up": a mechanism program would fit motions that nobody measured.

## What you made, and what comes next

- `results/M21_model_picks.csv` and `results/M21_pick_differences.csv`: the models' picks and their differences
  from yours.
- `results/pq_picks/`: the pick files of your three PocketQuake runs.

M31 locates the Sangju earthquakes with picks like these; M51 turns PocketQuake's first motions into focal
mechanisms. Your cluster `PQ_SLUG` stays registered.

**Read more:** Zhu and Beroza (2019), PhaseNet; Zhu et al. (2025), PhaseNet+; Woollam et al. (2022), SeisBench.
The [reading list](../../../docs/reading_list.md) has the rest.

Next: M30 (travel times).